# Emerging Technologies Assessment Summer 25/26

This notebook explores the difference between classical and quantum algorithms through a series of hands-on problems.
The problems are centred around the [Deutsch–Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa), a quantum algorithm that demonstrates an exponential speedup over its classical counterpart for a specific problem involving Boolean functions.

---

## Problem 1: Generating Random Boolean Functions

### Background

The [Deutsch–Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa) operates on Boolean functions of the form:

$$f: \{0,1\}^n \rightarrow \{0,1\}$$

where the function is **guaranteed** to be one of two kinds:

- **Constant:** The function returns the same output (`True` or `False`) for *every* possible input combination.
- **Balanced:** The function returns `True` for *exactly half* of all possible input combinations, and `False` for the other half.

With **four Boolean inputs**, there are $2^4 = 16$ possible input combinations.
This means a balanced function returns `True` for exactly **8** of those 16 inputs.

There are only **2 constant functions** (always-`False` and always-`True`), but $\binom{16}{8} = 12{,}870$ possible balanced functions — making a random balanced function effectively unique every time.

### Task

Write a Python function `random_constant_balanced` that randomly returns one function from the combined set of constant and balanced functions, each taking **four Boolean arguments**.

### Imports

Only modules from the Python standard library are needed for this problem:

- [`random`](https://docs.python.org/3/library/random.html) — for random selection of function type and truth-table assignments.
- [`itertools`](https://docs.python.org/3/library/itertools.html) — for generating all combinations of Boolean inputs efficiently.

In [1]:
import random      # Provides random choice and sampling utilities
import itertools   # Provides Cartesian product for enumerating all input combinations

### Approach

The strategy is to build a **truth table** — a dictionary that maps every possible input tuple to an output value — and then return a closure that looks up any query in that table.

1. Use `itertools.product([False, True], repeat=4)` to enumerate all 16 four-bit input combinations.
2. Randomly choose whether the function will be **constant** or **balanced**.
3. Fill the truth table accordingly:
   - *Constant:* Every input maps to the same randomly chosen value.
   - *Balanced:* Randomly sample 8 of the 16 inputs to map to `True`; the rest map to `False`.
4. Return an inner function `f(a, b, c, d)` that performs a single dictionary lookup.

In [2]:
def random_constant_balanced():
    """
    Return a randomly chosen constant or balanced Boolean function.

    The returned function accepts exactly four Boolean positional arguments
    (a, b, c, d) and returns a single Boolean value.

    - Constant: returns the same Boolean value for every possible input.
    - Balanced: returns True for exactly 8 of the 16 possible input combinations
      and False for the remaining 8.

    Returns
    -------
    function
        A callable f(a, b, c, d) -> bool backed by a pre-built truth table.
    """
    # Enumerate all 2^4 = 16 combinations of four Boolean inputs
    all_inputs = list(itertools.product([False, True], repeat=4))

    # Randomly decide whether this will be a constant or balanced function
    function_type = random.choice(["constant", "balanced"])

    if function_type == "constant":
        # Choose a single output value that every input will map to
        constant_output = random.choice([True, False])

        # Every input gets the same constant output
        truth_table = {inputs: constant_output for inputs in all_inputs}

    else:  # balanced
        # Randomly pick exactly 8 of the 16 inputs to return True
        true_inputs = set(random.sample(all_inputs, 8))

        # Inputs in the chosen set return True; all others return False
        truth_table = {inputs: (inputs in true_inputs) for inputs in all_inputs}

    def f(a, b, c, d):
        """Evaluate the Boolean function by looking up (a, b, c, d) in the truth table."""
        return truth_table[(a, b, c, d)]

    return f

### Demonstration

We call `random_constant_balanced()` to obtain a function, evaluate it on all 16 inputs, and display the resulting truth table.
By counting the number of `True` outputs we can verify which type was generated.

In [3]:
# All 16 possible four-bit input combinations (reused across demonstration cells)
all_inputs = list(itertools.product([False, True], repeat=4))

In [4]:
# Generate a single random constant-or-balanced function
func = random_constant_balanced()

# Evaluate the function on every possible input and store results
outputs = [func(a, b, c, d) for a, b, c, d in all_inputs]

# Count True outputs to identify the function type
true_count = sum(outputs)

# Classify based on the number of True outputs
if true_count == 0:
    detected_type = "Constant (always False)"
elif true_count == 16:
    detected_type = "Constant (always True)"
elif true_count == 8:
    detected_type = "Balanced"
else:
    detected_type = f"Unknown ({true_count} True outputs)"

print(f"Detected function type: {detected_type}")
print(f"True outputs: {true_count} / 16\n")

# Display the full truth table
print(f"{'a':>5}  {'b':>5}  {'c':>5}  {'d':>5}  |  {'f(a,b,c,d)':>10}")
print("-" * 46)
for (a, b, c, d), output in zip(all_inputs, outputs):
    print(f"{str(a):>5}  {str(b):>5}  {str(c):>5}  {str(d):>5}  |  {str(output):>10}")

Detected function type: Constant (always False)
True outputs: 0 / 16

    a      b      c      d  |  f(a,b,c,d)
----------------------------------------------
False  False  False  False  |       False
False  False  False   True  |       False
False  False   True  False  |       False
False  False   True   True  |       False
False   True  False  False  |       False
False   True  False   True  |       False
False   True   True  False  |       False
False   True   True   True  |       False
 True  False  False  False  |       False
 True  False  False   True  |       False
 True  False   True  False  |       False
 True  False   True   True  |       False
 True   True  False  False  |       False
 True   True  False   True  |       False
 True   True   True  False  |       False
 True   True   True   True  |       False


### References

Key external sources consulted while implementing Problem 1.

- [Deutsch–Jozsa algorithm — IBM Quantum Learning](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa)
  — defines the **constant** vs **balanced** classification of Boolean
  functions that drives the design of `random_constant_balanced`.
- [`random` — Python Standard Library](https://docs.python.org/3/library/random.html)
  — provides `random.choice` (picks the function type) and
  `random.sample` (selects the 8 inputs that map to `True` for balanced
  functions).
- [`itertools` — Python Standard Library](https://docs.python.org/3/library/itertools.html)
  — `itertools.product([False, True], repeat=4)` enumerates all 16
  four-bit input combinations used to build the truth table.

## Problem 2: Classical Testing for Function Type

### Background

[Deutsch's algorithm](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm) claims a quantum advantage over classical computation.
To appreciate that advantage, we must first establish the classical cost — i.e., the minimum number of times a classical algorithm must call `f` to determine its type with absolute certainty.

For a function `f` taking **four Boolean inputs**, there are $2^4 = 16$ possible input combinations.

| Function type | Condition |
|---|---|
| Constant | All 16 outputs are the same value (`True` or `False`) |
| Balanced | Exactly 8 outputs are `True` and 8 are `False` |

A classical algorithm cannot determine the type from a single call — the same output could come from either type.
The question is: **what is the minimum number of calls guaranteed to be sufficient?**

### Approach

The key insight is that we do **not** need to evaluate `f` on all 16 inputs.

- If two queries ever return **different values**, the function cannot be constant → return `"balanced"` immediately.
- If the same value is seen **9 times in a row**, the function cannot be balanced (a balanced function produces each value at most 8 times) → return `"constant"` immediately.

This gives us an early-exit strategy that needs **at most 9 calls** to reach a definitive answer.

In [5]:
def determine_constant_balanced(f):
    """
    Determine whether a Boolean function is constant or balanced.

    Queries f on successive distinct inputs and stops as soon as the type
    can be established with certainty.  At most 9 calls to f are ever needed.

    Parameters
    ----------
    f : callable
        A function accepting four Boolean positional arguments (a, b, c, d)
        and returning a single Boolean value.  The function is guaranteed to
        be either constant or balanced, as produced by random_constant_balanced.

    Returns
    -------
    str
        "constant" if f returns the same value for all 16 inputs, or
        "balanced" if f returns True for exactly 8 of the 16 inputs.
    """
    # All 16 four-bit input combinations in a fixed evaluation order
    all_inputs = list(itertools.product([False, True], repeat=4))

    # First query establishes the reference output value
    first_output = f(*all_inputs[0])

    # Query up to 8 more inputs (9 total); stop early if the type is clear
    # Reasoning: a balanced function has exactly 8 True and 8 False outputs,
    # so it cannot produce the same value 9 times — that would require 9 of one
    # value, leaving only 7 of the other, violating the balanced constraint.
    for inputs in all_inputs[1:9]:
        output = f(*inputs)

        # A differing output proves the function is not constant → balanced
        if output != first_output:
            return "balanced"

    # Nine consecutive identical outputs → impossible for balanced → constant
    return "constant"

### Demonstration

We generate functions using `random_constant_balanced` from Problem 1, then classify each one with `determine_constant_balanced`.
Ground truth is obtained by evaluating all 16 inputs exhaustively so we can confirm the classifier is always correct.

In [6]:
# Run several trials to verify the classifier against known ground truth
num_trials = 10

print(f"Running {num_trials} classification trials...\n")
print(f"{'Trial':>5}  {'Ground Truth':>14}  {'Classified As':>14}  {'Match':>6}")
print("-" * 46)

for trial in range(1, num_trials + 1):
    # Generate a fresh random constant-or-balanced function
    test_func = random_constant_balanced()

    # Determine ground truth by evaluating all 16 outputs exhaustively
    true_outputs = [test_func(a, b, c, d) for a, b, c, d in all_inputs]
    true_count = sum(true_outputs)

    # Classify ground truth: 0 or 16 Trues → constant, 8 Trues → balanced
    ground_truth = "constant" if true_count in (0, 16) else "balanced"

    # Classify using our efficient function (at most 9 calls)
    classification = determine_constant_balanced(test_func)

    # Flag whether the two labels agree
    match = "YES" if classification == ground_truth else "NO"

    print(f"{trial:>5}  {ground_truth:>14}  {classification:>14}  {match:>6}")

Running 10 classification trials...

Trial    Ground Truth   Classified As   Match
----------------------------------------------
    1        constant        constant     YES
    2        balanced        balanced     YES
    3        constant        constant     YES
    4        balanced        balanced     YES
    5        balanced        balanced     YES
    6        balanced        balanced     YES
    7        balanced        balanced     YES
    8        balanced        balanced     YES
    9        constant        constant     YES
   10        constant        constant     YES


### Efficiency

| Metric | Value |
|---|---|
| Maximum calls to `f` | **9** |
| Minimum calls to `f` | **2** |
| Total possible inputs | 16 |

**Why 9?**  
A balanced function has exactly **8** `True` outputs and **8** `False` outputs across all 16 inputs.  
If the first 8 queries all return the same value, it is still *possible* that the function is balanced — those 8 queries could have landed on exactly the 8 inputs where a balanced function agrees.  
Once a **9th query** returns the same value, there are now 9 identical outputs, which is impossible for a balanced function (it can produce at most 8 of each value). The function must therefore be constant.

**Why not all 16?**  
In the worst case — a constant function — the algorithm terminates after 9 calls instead of 16, saving nearly half the work compared to a naïve full scan.  
In the best case — any two consecutive queries disagree — it terminates in just **2 calls**.

This contrasts with [Deutsch's algorithm](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm) and the generalised [Deutsch–Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa), both of which always reach a definitive answer in **a single query**, regardless of the number of input bits.

### References

Key external sources consulted while implementing Problem 2.

- [Deutsch's algorithm — IBM Quantum Learning](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm)
  — establishes the quantum-vs-classical query advantage that the
  Efficiency analysis is benchmarked against (single quantum query vs
  up to 9 classical calls).
- [Deutsch–Jozsa algorithm — IBM Quantum Learning](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa)
  — n-bit generalisation referenced in the Efficiency section to show
  that the quantum advantage scales beyond the single-input case.
- [`itertools.product` — Python Standard Library](https://docs.python.org/3/library/itertools.html#itertools.product)
  — provides the deterministic input enumeration that drives the
  early-exit query loop in `determine_constant_balanced`.

## Problem 3: Quantum Oracles

### Background

In the **single-input** case there are exactly four Boolean functions of the form
$f:\{0,1\} \rightarrow \{0,1\}$:

| Name | $f(0)$ | $f(1)$ | Type |
|------|--------|--------|------|
| $f_0$ | 0 | 0 | Constant (always False) |
| $f_1$ | 0 | 1 | Balanced — identity |
| $f_2$ | 1 | 0 | Balanced — negation |
| $f_3$ | 1 | 1 | Constant (always True) |

[Deutsch's algorithm](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm)
determines in a single quantum query whether $f$ is constant or balanced.
To run the algorithm on a quantum computer, each classical function must first be
encoded as a **quantum oracle**.

### Quantum Oracle Definition

A quantum oracle encodes $f$ as a **unitary transformation** $U_f$ acting on two qubits:

$$U_f |x\rangle|y\rangle = |x\rangle|y \oplus f(x)\rangle$$

where:
- $|x\rangle$ (**qubit 0**, input register) — holds the query value and is **left unchanged**.
- $|y\rangle$ (**qubit 1**, ancilla register) — accumulates the function output via XOR ($\oplus$).

This design is **unitary**: applying $U_f$ twice returns to the original state ($U_f^2 = I$),
satisfying the reversibility required by quantum mechanics.

### Imports

- [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) from `qiskit` — builds oracle circuits gate by gate.
- [`Statevector`](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Statevector) from `qiskit.quantum_info` — simulates exact quantum states for oracle verification.
- `matplotlib.pyplot` — renders and saves circuit diagrams.
- `numpy` — constructs basis-state vectors for oracle testing.
- `os` — ensures the `img/` output directory exists.

In [7]:
# QuantumCircuit lets us build a quantum circuit one gate at a time.
from qiskit import QuantumCircuit

# Statevector gives an exact, noiseless simulation of a quantum state.
from qiskit.quantum_info import Statevector

# matplotlib renders the circuit diagrams and saves them as image files.
import matplotlib.pyplot as plt

# numpy is used to build the raw amplitude arrays for each basis state.
import numpy as np

# os is used to create the img/ output directory before saving figures.
import os

### Oracle Implementations

Each oracle is a 2-qubit circuit.
The gate sequence follows directly from
$U_f|x\rangle|y\rangle = |x\rangle|y \oplus f(x)\rangle$:

| Oracle | $f(0)$ | $f(1)$ | Gate sequence | Reasoning |
|--------|--------|--------|---------------|-----------|
| $f_0$: constant 0 | 0 | 0 | — (no gates) | $y \oplus 0 = y$ — ancilla is never modified |
| $f_1$: identity   | 0 | 1 | CNOT($q_0 \to q_1$) | $y \oplus x$ — flip ancilla only when input is $|1\rangle$ |
| $f_2$: negation   | 1 | 0 | X($q_0$), CNOT, X($q_0$) | $y \oplus \lnot x$ — flip ancilla only when input is $|0\rangle$ |
| $f_3$: constant 1 | 1 | 1 | X($q_1$) | $y \oplus 1 = \lnot y$ — always flip the ancilla |

In [8]:
def oracle_constant_0():
    """
    Build the quantum oracle for f(x) = 0 (constant zero).

    Implements Uf|x>|y> = |x>|y XOR 0> = |x>|y>.
    XORing with 0 leaves y unchanged, so the circuit needs no gates at all.

    Returns
    -------
    QuantumCircuit
        A 2-qubit circuit where q[0] is the input register and q[1] is the ancilla.
    """
    # Create an empty 2-qubit circuit because f(x)=0 means y XOR 0 = y.
    circuit = QuantumCircuit(2, name="f0: const-0")

    # Return the empty circuit, which acts as the identity on every input state.
    return circuit

In [9]:
def oracle_identity():
    """
    Build the quantum oracle for f(x) = x (identity function).

    Implements Uf|x>|y> = |x>|y XOR x>.
    A single CNOT gate with q[0] as control and q[1] as target encodes y XOR x exactly.

    Returns
    -------
    QuantumCircuit
        A 2-qubit circuit where q[0] is the input register and q[1] is the ancilla.
    """
    # Create a labelled 2-qubit circuit that will hold a single CNOT gate.
    circuit = QuantumCircuit(2, name="f1: identity")

    # CNOT flips the ancilla q[1] only when the input q[0] is |1>, giving y XOR x.
    circuit.cx(0, 1)

    # Return the finished oracle circuit.
    return circuit

In [10]:
def oracle_negation():
    """
    Build the quantum oracle for f(x) = NOT x (negation function).

    Implements Uf|x>|y> = |x>|y XOR (NOT x)>.
    Since f(0)=1 and f(1)=0, the ancilla must flip when the input is |0>.
    Surrounding the CNOT with X gates on q[0] inverts the control polarity, then
    restores q[0] so the input register is left untouched.

    Returns
    -------
    QuantumCircuit
        A 2-qubit circuit where q[0] is the input register and q[1] is the ancilla.
    """
    # Create a labelled 2-qubit circuit that will hold the X-CNOT-X pattern.
    circuit = QuantumCircuit(2, name="f2: negation")

    # Flip q[0] so the upcoming CNOT will fire when the original input was |0>.
    circuit.x(0)

    # CNOT flips q[1] when the (now flipped) q[0] is |1>, i.e. when original x = 0.
    circuit.cx(0, 1)

    # Flip q[0] back to its original value so the input register is unchanged.
    circuit.x(0)

    # Return the finished oracle circuit.
    return circuit

In [11]:
def oracle_constant_1():
    """
    Build the quantum oracle for f(x) = 1 (constant one).

    Implements Uf|x>|y> = |x>|y XOR 1> = |x>|NOT y>.
    XORing with 1 always flips the ancilla, so a single unconditional X gate suffices.

    Returns
    -------
    QuantumCircuit
        A 2-qubit circuit where q[0] is the input register and q[1] is the ancilla.
    """
    # Create a labelled 2-qubit circuit that will hold a single X gate on the ancilla.
    circuit = QuantumCircuit(2, name="f3: const-1")

    # X gate on q[1] flips the ancilla unconditionally, encoding y XOR 1 = NOT y.
    circuit.x(1)

    # Return the finished oracle circuit.
    return circuit

### Demonstration

We draw the circuit diagram for each oracle to show its gate structure and save
the diagrams to `img/` for reference.
The circuit depth reflects the gate count: $f_0$ uses no gates, $f_1$ and $f_3$
use one gate each, and $f_2$ uses three gates to implement the anti-CNOT pattern.

In [12]:
# Make sure the img/ output folder exists before saving any figures.
os.makedirs("img", exist_ok=True)

# Build all four oracle circuits in a labelled list so we can iterate over them.
oracle_display = [
    ("f0: const-0",  oracle_constant_0()),   # Constant zero — empty circuit.
    ("f1: identity", oracle_identity()),      # Identity — single CNOT.
    ("f2: negation", oracle_negation()),      # Negation — X, CNOT, X pattern.
    ("f3: const-1",  oracle_constant_1()),    # Constant one — single X on ancilla.
]

# Loop through every oracle and render its circuit diagram.
for label, circuit in oracle_display:
    # Use Qiskit's built-in matplotlib drawer to produce a figure of the circuit.
    fig = circuit.draw("mpl")

    # Add a title above the diagram so the oracle is easy to identify.
    fig.suptitle(label, fontsize=11)

    # Replace any spaces and colons in the label so it works as a filename.
    safe_name = label.replace(": ", "_").replace(" ", "_")

    # Save the diagram as a high-resolution PNG inside the img/ folder.
    fig.savefig(f"img/problem3_{safe_name}.png", dpi=150, bbox_inches="tight")

    # Display the figure inline in the notebook.
    plt.show()

### Verification

We verify each oracle by simulating it on all four computational basis states
$|x\rangle|y\rangle \in \{|0,0\rangle,\,|0,1\rangle,\,|1,0\rangle,\,|1,1\rangle\}$
and confirming that the output matches $|x\rangle|y \oplus f(x)\rangle$.

[`Statevector`](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Statevector)
provides exact (noiseless) simulation without measurement — this lets us inspect the
full quantum state directly rather than estimating it from repeated shots.

Qiskit uses **little-endian** qubit ordering: for a 2-qubit circuit the statevector
index of basis state $|q_1,\,q_0\rangle$ is $q_0 + 2\,q_1$, so $|x,y\rangle$
maps to index $x + 2y$.

**Why four basis states are sufficient.** Quantum operators are linear, so an
operator on 2 qubits is fully determined by its action on the four computational
basis states. If $U_f$ behaves correctly on each basis state, linearity guarantees
it behaves correctly on every superposition — including the $|+\rangle|-\rangle$
input that Deutsch's algorithm feeds it in Problem 4.

In [13]:
# Pair each oracle with its expected classical truth table for verification.
oracle_specs = [
    (oracle_constant_0(), {0: 0, 1: 0}),  # f0(x) = 0 for every input.
    (oracle_identity(),   {0: 0, 1: 1}),  # f1(x) = x (identity).
    (oracle_negation(),   {0: 1, 1: 0}),  # f2(x) = NOT x (negation).
    (oracle_constant_1(), {0: 1, 1: 1}),  # f3(x) = 1 for every input.
]

# Print a heading describing the property we are about to verify.
print("Verifying oracle transformation: Uf|x,y> = |x, y XOR f(x)>\n")
print("=" * 58)

# Loop over each oracle alongside its expected classical truth table.
for circuit, f_values in oracle_specs:

    # Print a header showing which oracle is being tested and its truth table.
    print(f"\nOracle: {circuit.name}   "
          f"[f(0)={f_values[0]}, f(1)={f_values[1]}]")

    # Print column titles for the per-input verification table.
    print(f"  {'Input':>8}  ->  {'Expected':>10}  {'Result':>10}  {'Pass':>5}")
    print("  " + "-" * 44)

    # Track whether every basis-state test for this oracle passes.
    all_pass = True

    # Iterate over both possible input qubit values.
    for x in [0, 1]:

        # Iterate over both possible ancilla starting values.
        for y in [0, 1]:

            # Allocate a 4-element complex array to represent the 2-qubit state.
            state_vec = np.zeros(4, dtype=complex)

            # Place all amplitude on |x, y> using Qiskit's little-endian index x + 2*y.
            state_vec[x + 2 * y] = 1.0

            # Wrap the raw array as a Qiskit Statevector so the circuit can act on it.
            sv = Statevector(state_vec)

            # Evolve the input state through the oracle circuit.
            sv_out = sv.evolve(circuit)

            # Compute the expected ancilla value as y XOR f(x).
            y_out = y ^ f_values[x]

            # Compute the index of the expected basis state in the statevector.
            expected_idx = x + 2 * y_out

            # Read the probability distribution of the post-oracle state.
            probs = sv_out.probabilities()

            # Identify which basis state actually carries the full probability mass.
            actual_idx = int(np.argmax(probs))

            # Decode the actual basis index back into (input bit, ancilla bit).
            actual_x, actual_y = actual_idx % 2, actual_idx // 2

            # Confirm that the expected basis state holds essentially all the probability.
            passed = abs(probs[expected_idx] - 1.0) < 1e-9

            # Update the running pass flag for this oracle.
            all_pass = all_pass and passed

            # Build a YES/NO marker for the per-row pass status.
            mark = "YES" if passed else "NO"

            # Format each ket label so it can share the same column width as the header.
            input_str = f"|{x},{y}>"
            expected_str = f"|{x},{y_out}>"
            result_str = f"|{actual_x},{actual_y}>"

            # Print the row using the same widths as the column headers above.
            print(f"  {input_str:>8}  ->  {expected_str:>10}  "
                  f"{result_str:>10}  {mark:>5}")

    # Summarise the oracle's overall verification outcome.
    summary = "ALL PASS" if all_pass else "FAILED"
    print(f"\n  Result: {summary}")

Verifying oracle transformation: Uf|x,y> = |x, y XOR f(x)>


Oracle: f0: const-0   [f(0)=0, f(1)=0]
     Input  ->    Expected      Result   Pass
  --------------------------------------------
     |0,0>  ->       |0,0>       |0,0>    YES
     |0,1>  ->       |0,1>       |0,1>    YES
     |1,0>  ->       |1,0>       |1,0>    YES
     |1,1>  ->       |1,1>       |1,1>    YES

  Result: ALL PASS

Oracle: f1: identity   [f(0)=0, f(1)=1]
     Input  ->    Expected      Result   Pass
  --------------------------------------------
     |0,0>  ->       |0,0>       |0,0>    YES
     |0,1>  ->       |0,1>       |0,1>    YES
     |1,0>  ->       |1,1>       |1,1>    YES
     |1,1>  ->       |1,0>       |1,0>    YES

  Result: ALL PASS

Oracle: f2: negation   [f(0)=1, f(1)=0]
     Input  ->    Expected      Result   Pass
  --------------------------------------------
     |0,0>  ->       |0,1>       |0,1>    YES
     |0,1>  ->       |0,0>       |0,0>    YES
     |1,0>  ->       |1,0>       |1,0> 

### How Each Oracle Works

The verification above confirms that all four oracles correctly implement
$U_f|x\rangle|y\rangle = |x\rangle|y \oplus f(x)\rangle$:

- **$f_0$ (constant 0):** The empty circuit leaves both qubits unchanged.  
  $y \oplus 0 = y$, so the ancilla is never modified and the input is trivially preserved.

- **$f_1$ (identity):** The CNOT flips the ancilla when and only when $x = 1$.  
  For $x=0$: ancilla is unchanged ($y \oplus 0$). For $x=1$: ancilla is flipped ($y \oplus 1$).

- **$f_2$ (negation):** The anti-CNOT pattern (X, CNOT, X on $q_0$) flips the ancilla
  when and only when $x = 0$ — the opposite polarity of $f_1$.  
  The flanking X gates convert a $|0\rangle$-controlled flip into a standard CNOT operation.

- **$f_3$ (constant 1):** A single X gate on the ancilla flips it unconditionally.  
  $y \oplus 1 = \lnot y$, regardless of $x$. The input register is untouched.

These four circuits are the building blocks for
[Deutsch's algorithm](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm)
in **Problem 4**: each oracle is embedded inside a circuit that wraps it between
Hadamard layers, enabling a single measurement to distinguish constant from balanced.

### References

Key external sources consulted while implementing Problem 3.

- [Deutsch's algorithm — IBM Quantum Learning](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm)
  — motivates the oracle construction; the four oracles built here are
  consumed by Deutsch's algorithm in Problem 4.
- [`QuantumCircuit` — Qiskit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit)
  — API reference for the gate methods (`.x`, `.cx`, `.draw`) used to
  build all four oracles and render the diagrams saved to `img/`.
- [`Statevector` — Qiskit](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Statevector)
  — exact (noiseless) state simulator used in the Verification section
  to confirm $U_f|x,y\rangle = |x, y \oplus f(x)\rangle$ on every basis
  state; chosen over a shot-based simulator because the basis-state
  proof must be exact, not statistical.

## Problem 4: Deutsch's Algorithm with Qiskit

### Background

[Deutsch's algorithm](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm)
is the first quantum algorithm proven to outperform every classical algorithm
on a concrete computational problem.

The problem is the single-input version of the Deutsch–Jozsa problem: given
oracle access to a function $f:\{0,1\} \rightarrow \{0,1\}$, determine whether
$f$ is **constant** or **balanced** with as few calls to the oracle as possible.

| Algorithm | Calls to $f$ required | Certainty |
|-----------|----------------------|-----------|
| Classical (deterministic) | **2** in the worst case | 100% |
| Deutsch (quantum) | **1** always | 100% |

A classical algorithm cannot decide from a single call: the same output value
is consistent with both a constant and a balanced function.
Deutsch's algorithm uses [quantum superposition](https://scienceexchange.caltech.edu/topics/quantum-science-explained/quantum-superposition)
and interference to evaluate a
[global property](https://plato.stanford.edu/entries/qt-entangle/#5)
of $f$ — constant or balanced — in a single oracle query.

### Circuit Design

The algorithm uses two qubits:

- **q[0]** (input register) — queried in superposition to encode both inputs at once.
- **q[1]** (ancilla register) — initialised to $|1\rangle$ to enable **phase kickback**.

**Step-by-step circuit:**

| Step | Gate(s) | State after step |
|------|---------|------------------|
| 0 | — | $|0\rangle|0\rangle$ |
| 1 | X on q[1] | $|0\rangle|1\rangle$ |
| 2 | H on q[0] and q[1] | $|{+}\rangle|{-}\rangle$ |
| 3 | Oracle $U_f$ | $\frac{1}{\sqrt{2}}\bigl[(-1)^{f(0)}|0\rangle + (-1)^{f(1)}|1\rangle\bigr]|{-}\rangle$ |
| 4 | H on q[0] | $|f(0)\oplus f(1)\rangle|{-}\rangle$ (up to global phase) |
| 5 | Measure q[0] | $0$ → constant, $\;1$ → balanced |

**Phase kickback** is the key mechanism.
When the ancilla is in state $|{-}\rangle = \frac{1}{\sqrt{2}}(|0\rangle-|1\rangle)$,
the oracle imprints $f(x)$ as a phase on the input qubit rather than flipping the ancilla:

$$U_f|x\rangle|{-}\rangle = (-1)^{f(x)}|x\rangle|{-}\rangle$$

After the oracle, the ancilla remains $|{-}\rangle$ and the input qubit holds the phase pattern:

- **Constant** $f(0)=f(1)$: both amplitudes carry the same sign
  $\Rightarrow$ input is in $|{+}\rangle$ (up to global phase)
  $\Rightarrow$ H maps it to $|0\rangle$.
- **Balanced** $f(0)\ne f(1)$: amplitudes carry opposite signs
  $\Rightarrow$ input is in $|{-}\rangle$ (up to global phase)
  $\Rightarrow$ H maps it to $|1\rangle$.

The final H gate converts the relative phase difference into a measurable amplitude
difference — this is **quantum interference**.

### Imports

Problem 4 introduces two new imports from the Qiskit ecosystem:

- [`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html) from `qiskit_aer` — a local noiseless quantum circuit simulator;
  no IBM Quantum account or internet connection is required.
- [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile) from `qiskit` — compiles a circuit to the simulator's native gate set before execution.

In [14]:
from qiskit_aer import AerSimulator   # Local noiseless quantum circuit simulator
from qiskit import transpile           # Compiles circuits to the simulator's native gate set

In [15]:
def deutsch_circuit(oracle):
    """
    Build Deutsch's algorithm circuit that wraps a single oracle query.

    The circuit uses phase kickback to encode the function's type (constant or
    balanced) as a relative phase on the input qubit, then reads it with a
    Hadamard gate.  Exactly one call to the oracle is sufficient.

    Parameters
    ----------
    oracle : QuantumCircuit
        A 2-qubit oracle implementing Uf|x>|y> = |x>|y XOR f(x)>.
        q[0] is the input register; q[1] is the ancilla.

    Returns
    -------
    QuantumCircuit
        Complete Deutsch circuit with one classical bit for the measurement result.
        Measurement outcome 0 means constant; 1 means balanced.
    """
    # Two qubits (input + ancilla) and one classical bit for the measurement outcome
    circuit = QuantumCircuit(2, 1, name="Deutsch")

    # Setup
    # Set ancilla to |1>; combined with the next H gate this creates |-> for kickback
    circuit.x(1)
    # Put input into equal superposition |+> and ancilla into |->
    circuit.h(0)
    circuit.h(1)

    # Visual separator — marks the boundary between setup and oracle in the diagram
    circuit.barrier()

    # Oracle query
    # Treat the oracle as a black-box gate — makes the single-query nature explicit
    circuit.append(oracle.to_gate(), [0, 1])

    # Visual separator — marks the boundary between oracle and readout
    circuit.barrier()

    # Readout
    # Hadamard converts relative phase to amplitude: |+> -> |0> (constant), |-> -> |1> (balanced)
    circuit.h(0)
    # Measure the input qubit into the classical bit: 0 = constant, 1 = balanced
    circuit.measure(0, 0)

    return circuit

### Demonstration

We build Deutsch's circuit for each of the four oracles from Problem 3 and draw
the resulting diagrams. Each diagram shows the same three-stage structure
(setup / oracle / readout) separated by barrier markers.
Only the oracle box in the middle changes — this makes the single-query nature
of the algorithm visually clear.

In [16]:
# Build one Deutsch circuit per oracle and store them for drawing and simulation
deutsch_circuits = [
    ("f0: const-0",  deutsch_circuit(oracle_constant_0())),
    ("f1: identity", deutsch_circuit(oracle_identity())),
    ("f2: negation", deutsch_circuit(oracle_negation())),
    ("f3: const-1",  deutsch_circuit(oracle_constant_1())),
]

# Draw and save each circuit diagram — oracle appears as a labelled black-box gate
for label, circuit in deutsch_circuits:
    fig = circuit.draw("mpl")
    fig.suptitle(f"Deutsch's Algorithm — Oracle {label}", fontsize=11)
    safe_name = label.replace(": ", "_").replace(" ", "_")
    fig.savefig(f"img/problem4_deutsch_{safe_name}.png", dpi=150, bbox_inches="tight")
    plt.show()

### Running the Algorithm

Each Deutsch circuit is executed on the `AerSimulator` using 1024 shots.

Deutsch's algorithm is **deterministic** — the result is always certain, not
probabilistic. Every shot should produce the same outcome:

- Measurement `'0'` — function is **constant** (both amplitudes interfere constructively at $|0\rangle$).
- Measurement `'1'` — function is **balanced** (amplitudes interfere constructively at $|1\rangle$).

A 100 % count in one state confirms the deterministic correctness of the algorithm.

In [17]:
# Noiseless local simulator — no IBM Quantum account required
simulator = AerSimulator()

# Ground-truth labels: what outcome and type each oracle should produce
expected = {
    "f0: const-0":  ("0", "constant"),
    "f1: identity": ("1", "balanced"),
    "f2: negation": ("1", "balanced"),
    "f3: const-1":  ("0", "constant"),
}

print("Running Deutsch's algorithm on all four oracles (1024 shots each)\n")
print(f"{'Oracle':>14}  {'Counts':>18}  {'Result':>10}  {'Expected':>10}  {'Pass':>5}")
print("-" * 64)

for label, circuit in deutsch_circuits:
    # Compile the circuit to the simulator's native instruction set
    compiled = transpile(circuit, simulator)
    # Execute the circuit and collect measurement counts over all shots
    job = simulator.run(compiled, shots=1024)
    counts = job.result().get_counts()

    # Determine the algorithm's answer from the dominant measurement outcome
    outcome = max(counts, key=counts.get)
    result_label = "constant" if outcome == "0" else "balanced"

    exp_outcome, exp_label = expected[label]
    passed = "YES" if outcome == exp_outcome else "NO"

    # Format counts as '0': XXXX  '1': YYYY for readability
    counts_str = f"'0':{counts.get('0', 0):>5}  '1':{counts.get('1', 0):>5}"
    print(f"{label:>14}  {counts_str:>18}  {result_label:>10}  {exp_label:>10}  {passed:>5}")

Running Deutsch's algorithm on all four oracles (1024 shots each)

        Oracle              Counts      Result    Expected   Pass
----------------------------------------------------------------
   f0: const-0  '0': 1024  '1':    0    constant    constant    YES
  f1: identity  '0':    0  '1': 1024    balanced    balanced    YES
  f2: negation  '0':    0  '1': 1024    balanced    balanced    YES
   f3: const-1  '0': 1024  '1':    0    constant    constant    YES


### Interference Pattern

The measurement result is a direct consequence of how the final Hadamard gate
processes the relative phase pattern created by the oracle.

| Oracle | Phase pattern after $U_f$ | Input qubit state | After H on q[0] | Measurement |
|--------|---------------------------|-------------------|-----------------|-------------|
| $f_0$: const-0 | $+|0\rangle + |1\rangle$ | $|{+}\rangle$ | $|0\rangle$ | **0** |
| $f_1$: identity | $+|0\rangle - |1\rangle$ | $|{-}\rangle$ | $|1\rangle$ | **1** |
| $f_2$: negation | $-|0\rangle + |1\rangle$ | $-|{-}\rangle$ | $-|1\rangle$ | **1** |
| $f_3$: const-1 | $-|0\rangle - |1\rangle$ | $-|{+}\rangle$ | $-|0\rangle$ | **0** |

*(Global phase factors of $\frac{1}{\sqrt{2}}$ omitted for clarity;
global minus signs are physically unobservable and do not affect measurement.)*

**Why the Hadamard gate reveals the answer:**

The H gate transforms $|{+}\rangle \leftrightarrow |0\rangle$ and $|{-}\rangle \leftrightarrow |1\rangle$.
Constant functions leave the input in $|{+}\rangle$ (both amplitudes same sign) — constructive
interference at $|0\rangle$ and destructive at $|1\rangle$.
Balanced functions leave the input in $|{-}\rangle$ (amplitudes opposite sign) — the pattern reverses.

**Why this is impossible classically:**

A classical algorithm evaluates $f$ at a single input and observes one output value.
That value is consistent with both a constant and a balanced function, so a second
call is always necessary for certainty.
Deutsch's algorithm avoids this by never evaluating $f$ at a single point — it
evaluates a global interference pattern that encodes whether $f(0) = f(1)$ or
$f(0) \ne f(1)$, which is exactly the constant-vs-balanced distinction.

### References

Key external sources consulted while implementing Problem 4.

- [Deutsch's algorithm — IBM Quantum Learning](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm)
  — canonical specification of the algorithm steps; source for the
  state-evolution table in the Circuit Design section.
- [Quantum circuits — IBM Quantum Learning](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/quantum-circuits/introduction)
  — primary reference for the Qiskit conventions used to build the
  Deutsch circuit (qubit indexing, barriers, measurement).
- [`AerSimulator` — Qiskit Aer documentation](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)
  — high-performance local simulator used for the 1024-shot
  demonstration; chosen over `Statevector` (used in Problem 3) because
  the brief asks to *demonstrate* the circuit on real measurement
  outcomes.
- [`transpile` — Qiskit](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile)
  — compiles the circuit to the simulator's native gate set before
  execution.
- [Quantum superposition — Caltech Science Exchange](https://scienceexchange.caltech.edu/topics/quantum-science-explained/quantum-superposition)
  — accessible explanation of the $|+\rangle$ and $|-\rangle$ states
  that the Hadamard gates create at the start of the algorithm.
- [Global properties — Stanford Encyclopedia of Philosophy](https://plato.stanford.edu/entries/qt-entangle/#5)
  — frames "constant vs balanced" as a *global property* of $f$ — the
  feature that the interference pattern reads in a single query.

## Problem 5: Scaling to the Deutsch–Jozsa Algorithm

### Background

In Problem 4 we solved Deutsch's algorithm for a function with **one input bit**.
The [Deutsch–Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa)
does the same thing but for a function with **four input bits** (or any number of bits).

The task is still the same: given a function that is guaranteed to be either
**constant** (always returns the same value) or **balanced** (returns `True` for exactly
half of all inputs), figure out which type it is.

Here is why quantum wins:

| Algorithm | Calls needed | Certain? |
|-----------|-------------|----------|
| Classical (Problem 2) | Up to **9** calls | Yes |
| Deutsch–Jozsa (quantum) | Always **1** call | Yes |

The quantum algorithm checks **all 16 inputs at once** by putting the qubits into
superposition, then uses interference to read a global property (constant vs. balanced)
from a single measurement.

### How the Circuit Works

The circuit uses **5 qubits**: four input qubits ($q_0$–$q_3$) and one helper qubit
called the **ancilla** ($q_4$).

| Step | What happens |
|------|--------------|
| 1 | Flip the ancilla to $|1\rangle$ with an X gate |
| 2 | Apply H to all 5 qubits — puts every qubit into superposition |
| 3 | Run the oracle — it secretly "marks" each input based on $f$ |
| 4 | Apply H to the 4 input qubits again |
| 5 | Measure: all zeros (`'0000'`) → **constant**; anything else → **balanced** |

The key trick is called **phase kickback**.
When the ancilla is in state $|{-}\rangle$, the oracle does not flip it.
Instead, it leaves a $+$ or $-$ sign on each input state depending on whether $f(x) = 0$ or $1$.
The final H gates turn those signs into measurable 0s and 1s:

- **Constant function** — all signs agree → they add together at `'0000'` → you always measure `'0000'`.
- **Balanced function** — half the signs are $+$ and half are $-$ → they cancel at `'0000'` → you never measure `'0000'`.

### Building the Oracle

The oracle is a quantum circuit that encodes the classical function $f$.
Its job is: for each input $x$, if $f(x) = 1$, flip the ancilla qubit.

$$U_f \, |x\rangle|y\rangle = |x\rangle|y \oplus f(x)\rangle$$

To build this, we use a **multi-controlled X (MCX)** gate.
An MCX gate flips a target qubit **only when all control qubits are** $|1\rangle$.

We loop over all 16 inputs. For each input where $f = 1$:

1. For every input bit that is `False` (0), add an **X gate** to temporarily flip that qubit to $|1\rangle$.
2. Add an **MCX gate** — this flips the ancilla when all four input qubits are $|1\rangle$.
3. Add the **same X gates again** to flip those qubits back to their original values.

**Simple example:** if $f(\text{False, False, False, False}) = 1$, we need to flip the ancilla
when all inputs are 0.
We add X gates on all 4 input qubits (0 → 1), run the MCX, then add the X gates again (1 → 0).

The algorithm only **calls this oracle once**, no matter how complex it is inside.

**Note on `constant_true`.** The generic builder emits 16 X–MCX–X blocks for $f(x)=1$
(one per input). A specialised circuit would replace this with a single X on the ancilla
and produce the same overall unitary. We keep the generic form because the brief asks for
one circuit that handles **any** four-bit Boolean function — clarity over compactness.

**Why testing on superposition inputs is correct.** Quantum gates are linear, so an oracle
that correctly maps every computational basis state $|x\rangle|y\rangle$ to
$|x\rangle|y \oplus f(x)\rangle$ automatically maps every superposition
$\sum_x \alpha_x |x\rangle|y\rangle$ to $\sum_x \alpha_x |x\rangle|y \oplus f(x)\rangle$.
This is what lets the algorithm encode all 16 function values in a single oracle call.

In [18]:
def build_dj_oracle(f):
    """
    Build a 5-qubit quantum oracle for the Deutsch-Jozsa algorithm.

    Converts a classical Boolean function into a unitary that implements
    Uf|x0,x1,x2,x3>|y> = |x0,x1,x2,x3>|y XOR f(x0,x1,x2,x3)>.

    For every input where f is True, a multi-controlled X (MCX) gate flips
    the ancilla.  X gates before and after each MCX act as anti-controls,
    enabling the gate to fire on zero-valued input bits.

    Parameters
    ----------
    f : callable
        A function f(a, b, c, d) -> bool, as produced by random_constant_balanced.

    Returns
    -------
    QuantumCircuit
        5-qubit circuit: q[0]-q[3] are input registers; q[4] is the ancilla.
    """
    # 4 input qubits (q[0]-q[3]) + 1 ancilla qubit (q[4])
    circuit = QuantumCircuit(5, name="DJ Oracle")

    # All 16 four-bit input combinations in a fixed order
    all_inputs = list(itertools.product([False, True], repeat=4))

    for inputs in all_inputs:
        # f=False means y XOR 0 = y; ancilla is unchanged so skip this input
        if not f(*inputs):
            continue

        # Flip every qubit that is False so all MCX controls see |1>
        for qubit_idx, bit in enumerate(inputs):
            if not bit:
                circuit.x(qubit_idx)

        # MCX fires when all four input qubits are |1>, flipping the ancilla q[4]
        circuit.mcx([0, 1, 2, 3], 4)

        # Restore any flipped qubits to their original values (uncompute)
        for qubit_idx, bit in enumerate(inputs):
            if not bit:
                circuit.x(qubit_idx)

    return circuit

### The Deutsch–Jozsa Circuit

The circuit follows the same three-stage pattern as Problem 4, just with more qubits:

1. **Setup** — Flip the ancilla ($q_4$) with X, then apply H to all 5 qubits to create superposition.
2. **Oracle** — One call to $U_f$, shown as a single labelled box in the diagram.
3. **Readout** — Apply H to the 4 input qubits, then measure them.

The oracle is drawn as a **black box** so you can clearly see that only one query is made.
This is exactly what gives the quantum algorithm its advantage over the classical approach.

In [19]:
def deutsch_jozsa_circuit(oracle):
    """
    Build the Deutsch-Jozsa circuit for a four-bit Boolean function.

    The ancilla is initialised to |-> so the oracle encodes f as phases on
    the input register (phase kickback).  The final Hadamard layer converts
    the phase pattern to measurable amplitudes: outcome "0000" means constant,
    any non-zero outcome means balanced.

    Parameters
    ----------
    oracle : QuantumCircuit
        5-qubit oracle implementing Uf|x>|y> = |x>|y XOR f(x)>.

    Returns
    -------
    QuantumCircuit
        Full Deutsch-Jozsa circuit; measurement "0000" -> constant, else balanced.
    """
    # 5 qubits (4 input + 1 ancilla) and 4 classical bits for the output
    circuit = QuantumCircuit(5, 4, name="Deutsch-Jozsa (n=4)")

    # Setup
    # Flip ancilla to |1> so H will produce |-> for phase kickback
    circuit.x(4)
    # H on all qubits: input qubits enter equal superposition, ancilla becomes |->
    circuit.h(range(5))

    circuit.barrier()

    # Oracle query
    # One oracle call encodes all 16 function values simultaneously via superposition
    circuit.append(oracle.to_gate(), range(5))

    circuit.barrier()

    # Readout
    # H converts relative phases to measurable amplitudes in the Z basis
    circuit.h(range(4))
    # Measure: "0000" = constant function, any other result = balanced function
    circuit.measure(range(4), range(4))

    return circuit